# 项目四：Schwartz–Zippel 与随机验证

**组号：**　　**成员：**　　**分工：**

本项目先枚举有限域网格上的零点，再实现 Freivalds 矩阵乘法核验。重点是区分理论上界与经验概率，并理解单侧错误和独立重复。

## 1. 网格零点比例

若非零多项式的总次数不超过 $d$，Schwartz–Zippel 引理给出 $\Pr[f(r)=0]\leq\min\{1,d/|S|\}$。

In [ ]:
def count_zeros_2d(f, p):
    count = 0
    # TODO 1：枚举 F_p^2，统计 f(x,y)=0 的点
    return count


def sz_report(name, f, degree, p):
    zero_count = count_zeros_2d(f, p)
    return {
        'name': name,
        'zero_count': zero_count,
        'actual_rate': zero_count / (p * p),
        'sz_bound': min(1, degree / p),
    }


p = 7
examples = [
    ('x+y', lambda x, y: x + y, 1),
    ('x(x-1)', lambda x, y: x * (x - 1), 2),
    ('x^7-x', lambda x, y: x**7 - x, 7),
]

for name, f, degree in examples:
    print(sz_report(name, f, degree, p))

assert count_zeros_2d(examples[0][1], p) == 7
assert count_zeros_2d(examples[1][1], p) == 14
assert count_zeros_2d(examples[2][1], p) == 49

记录哪些例子达到上界。解释 $x^7-x$ 为什么是非零形式多项式，却在 $\mathbb F_7$ 上处处取零，以及此时上界为什么没有排除作用。

## 2. Freivalds 矩阵乘法核验

若 $AB=C$，则每个向量 $r$ 都满足 $A(Br)=Cr$；若 $AB\ne C$，随机向量有较大概率暴露错误。返回 `True` 仅表示暂时接受，不等于确定正确。

In [ ]:
import random

def mat_vec(M, v, p):
    return [sum(a * b for a, b in zip(row, v)) % p for row in M]


def freivalds(A, B, C, p, rounds=1, rng=None):
    if rng is None:
        rng = random
    n = len(A)
    for _ in range(rounds):
        r = [rng.randrange(p) for _ in range(n)]
        Br = mat_vec(B, r, p)
        ABr = mat_vec(A, Br, p)
        Cr = mat_vec(C, r, p)
        # TODO 2：发现 ABr != Cr 时立即返回 False
        pass
    # TODO 3：所有轮次都未发现错误时返回 True
    pass

In [ ]:
A = [[1, 0], [0, 1]]
B = [[1, 2], [0, 1]]
C_correct = [[1, 2], [0, 1]]
C_wrong = [[1, 2], [1, 1]]

for seed in range(20):
    assert freivalds(A, B, C_correct, 7, rounds=3, rng=random.Random(seed))
print('正确乘积始终通过测试')

# 错误结果有时会通过一轮检查；多运行几次观察
for seed in range(10):
    print(seed, freivalds(A, B, C_wrong, 7, rounds=1, rng=random.Random(seed)))

## 3. 估计漏检概率

In [ ]:
def estimate_false_acceptance(A, B, C, p, rounds, trials=10000, seed=2026):
    rng = random.Random(seed)
    accepted = 0
    # TODO 4：重复 trials 次，统计错误结果被接受的比例
    return accepted / trials


records = []
for rounds in [1, 2, 3, 5]:
    rate = estimate_false_acceptance(A, B, C_wrong, 7, rounds)
    bound = (1 / 7) ** rounds
    records.append((rounds, rate, bound))

print('轮数  经验漏检率  理论值/上界')
for record in records:
    print(record)

## 4. 边界与扩展实验

任选一项完成：

1. 把随机向量坐标限制在 $\{0,1\}$，修改程序并与 $2^{-k}$ 比较；
2. 修改 `C_wrong` 中不同位置，观察漏检概率是否相同；
3. 构造 $3\times3$ 矩阵实例并重复实验。

在下面增加代码单元，保留输出并解释结果。

## 5. 结果分析（请填写）

1. 哪些零点实验达到 SZ 上界？理论上界是否总等于真实概率？
2. 为什么正确的矩阵乘积永远不会被拒绝？
3. 为什么一次检查通过不能证明 $AB=C$？
4. 重复检查为什么要求每轮重新、独立地产生随机向量？
5. 用 100–200 字总结低次数多项式根数界与随机验证之间的关系。